In [ ]:
import pandas as pd
df = pd.read_csv("../data/groceries - groceries.csv")

In [ ]:
df.head()

In [ ]:
print(df.columns)
print(df.shape)
df.info()

In [ ]:
df.isnull().sum()

## Task 1: Converting Wide Tabular Data into Transaction Baskets

Here we extract the item columns (`Item 1` through `Item 32`) and convert each transaction row into a clean, lowercased, deduplicated list of purchased grocery items (ignoring `NaN`s).

In [ ]:
# Select only the item columns (excluding 'Item(s)' count)
item_cols = [col for col in df.columns if col.startswith("Item ")]

# Convert each transaction row into a clean, trimmed, deduplicated list of items
transactions = []
for row in df[item_cols].values:
    # Extract valid non-null items, strip spaces, convert to lowercase
    basket = [str(item).strip().lower() for item in row if pd.notna(item) and str(item).strip()]
    if basket:
        # Deduplicate items within the same basket using set()
        transactions.append(sorted(list(set(basket))))

print(f"Total Transactions Processed: {len(transactions)}")
print(f"Sample Basket 1: {transactions[0]}")
print(f"Sample Basket 2: {transactions[1]}")
print(f"Sample Basket 3: {transactions[2]}")

## Task 2: Catalog Profiling & Total Unique Products

Now that we have clean transaction baskets, let's explore the store catalog:
- Total items purchased across all shopping trips.
- Total unique products available in the store catalog ($|I|$).
- Average items purchased per transaction basket.
- A preview of unique products sorted alphabetically.

In [ ]:
# Flatten all items from every basket into a single list
all_items = [item for basket in transactions for item in basket]

# Extract unique product names
unique_products = sorted(list(set(all_items)))

total_items_sold = len(all_items)
total_unique_products = len(unique_products)
avg_basket_size = total_items_sold / len(transactions)

print("=== Store Catalog Summary ===")
print(f"Total Transactions:           {len(transactions):,}")
print(f"Total Items Sold:             {total_items_sold:,}")
print(f"Total Unique Products (SKUs): {total_unique_products}")
print(f"Average Basket Size:          {avg_basket_size:.2f} items")

print("\nSample of Unique Products (First 15 Alphabetically):")
for i, prod in enumerate(unique_products[:15], 1):
    print(f"{i:2d}. {prod}")

## Task 3: Analyzing & Plotting the Top 20 Most Frequent Products

In Market Basket Analysis, understanding which products are most frequent is essential:
- Frequency is directly proportional to single-item **Support**: $\text{Support}(X) = \frac{\text{Count}(X)}{N}$.
- Ubiquitous items (e.g. `whole milk`, `other vegetables`) have very high support (~19% to 25%).
- Knowing this distribution helps us choose an appropriate `min_support` threshold for FP-Growth and highlights why **Lift** is needed to prevent staple items from dominating every recommendation.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Count frequency of every purchased product
item_counts = Counter(all_items)

# Create DataFrame for Top 20 items
top_20_df = pd.DataFrame(item_counts.most_common(20), columns=["Product", "Frequency"])
top_20_df["Support (%)"] = (top_20_df["Frequency"] / len(transactions)) * 100

print("=== Top 20 Most Frequently Purchased Products ===")
print(top_20_df.to_string(index=False))

# Plot horizontal bar chart
plt.figure(figsize=(10, 7))
y_pos = range(len(top_20_df))
plt.barh(y_pos, top_20_df["Frequency"][::-1], color="#4f46e5", edgecolor="#312e81", height=0.65)
plt.yticks(y_pos, top_20_df["Product"][::-1], fontsize=11)
plt.xlabel("Transaction Frequency (Number of Baskets)", fontsize=12, labelpad=10)
plt.title("Top 20 Most Popular Grocery Products", fontsize=14, fontweight="bold", pad=15)
plt.grid(axis="x", linestyle="--", alpha=0.6)

# Add frequency and support labels on the bars
for i, (freq, supp) in enumerate(zip(top_20_df["Frequency"][::-1], top_20_df["Support (%)"][::-1])):
    plt.text(freq + 25, i, f"{freq:,} ({supp:.1f}%)", va="center", fontsize=9, color="#1e293b")

plt.xlim(0, top_20_df["Frequency"].max() * 1.18)
plt.tight_layout()
plt.show()

## Task 4: Basket Size Distribution & Summary Statistics

In this step, we analyze how many items customers purchase in a single shopping basket:
- What is the minimum, maximum, median, and average basket size?
- What percentage of transactions contain only **1 item** versus **multi-item baskets**?
- *Key Insight for Association Rules:* Single-item baskets cannot generate recommendations or bundles because an association rule requires at least an antecedent ($X$) and a consequent ($Y$), meaning $|basket| \ge 2$.

In [ ]:
import numpy as np

# Calculate the size (number of items) of each basket
basket_sizes = pd.Series([len(basket) for basket in transactions], name="Basket Size")

# Descriptive statistics
print("=== Basket Size Descriptive Statistics ===")
print(basket_sizes.describe())

# Single-item vs Multi-item baskets
single_item_count = (basket_sizes == 1).sum()
multi_item_count = (basket_sizes >= 2).sum()
pct_multi = (multi_item_count / len(basket_sizes)) * 100

print(f"\nSingle-Item Baskets (1 item):    {single_item_count:,} ({100 - pct_multi:.1f}%)")
print(f"Multi-Item Baskets (>= 2 items): {multi_item_count:,} ({pct_multi:.1f}%) -> Eligible for Bundles")

# Plot basket size distribution histogram
plt.figure(figsize=(10, 5))
max_size = basket_sizes.max()
bins = np.arange(1, max_size + 2) - 0.5

plt.hist(basket_sizes, bins=bins, color="#059669", edgecolor="#064e3b", rwidth=0.85)

# Highlight median and mean
median_val = basket_sizes.median()
mean_val = basket_sizes.mean()
plt.axvline(median_val, color="#dc2626", linestyle="--", linewidth=2, label=f"Median: {median_val:.0f} items")
plt.axvline(mean_val, color="#f59e0b", linestyle="-", linewidth=2, label=f"Mean: {mean_val:.2f} items")

plt.title("Distribution of Transaction Basket Sizes", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Number of Items in Basket", fontsize=12, labelpad=10)
plt.ylabel("Number of Transactions", fontsize=12, labelpad=10)
plt.xticks(range(1, min(max_size + 1, 33), 2))
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Task 5: Transaction Space & Matrix Sparsity Analysis

In retail point-of-sale data, **sparsity** measures the proportion of empty (unpurchased) cells if we represent transactions as a matrix where:
- **Rows** = Transactions ($N = 9,835$)
- **Columns** = Unique Products ($M = 169$)
- **Cell value** = $1$ (bought) or $0$ (not bought)

Understanding sparsity explains why brute-force candidate pair generation is computationally wasteful and why tree-based pattern compression (**FP-Growth**) is necessary.

In [ ]:
# Dimensions of the transactional space
n_transactions = len(transactions)
n_products = len(unique_products)
total_possible_cells = n_transactions * n_products
actual_items_purchased = len(all_items)

# Density and Sparsity calculations
density = (actual_items_purchased / total_possible_cells) * 100
sparsity = 100 - density

print("=== Transaction Matrix Space Analysis ===")
print(f"Total Transactions (Rows):      {n_transactions:,}")
print(f"Total Unique Products (Cols):   {n_products:,}")
print(f"Total Matrix Cells:             {total_possible_cells:,}")
print(f"Filled Cells (Purchases):       {actual_items_purchased:,} ({density:.2f}% density)")
print(f"Empty Cells (Non-purchases):    {total_possible_cells - actual_items_purchased:,} ({sparsity:.2f}% sparsity)")

# Visual representation of Density vs Sparsity
plt.figure(figsize=(6, 6))
plt.pie(
    [actual_items_purchased, total_possible_cells - actual_items_purchased],
    labels=[f"Purchased ({density:.2f}%)", f"Empty Cells ({sparsity:.2f}%)"],
    colors=["#10b981", "#e2e8f0"],
    explode=(0.1, 0),
    autopct="%1.1f%%",
    startangle=140,
    textprops={"fontsize": 11, "fontweight": "bold"},
    wedgeprops={"edgecolor": "#64748b", "linewidth": 1.2}
)
plt.title("Transaction Space Sparsity (97.4% Empty)", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

## Task 6: One-Hot Transaction Encoding with TransactionEncoder

The FP-Growth algorithm in `mlxtend` requires transactional data to be represented as a **binary incidence matrix** (one-hot encoded DataFrame):
- **Rows ($N = 9,835$):** Each row represents a single customer transaction.
- **Columns ($M = 169$):** Each column represents a unique grocery product.
- **Values:** `True` if the item was in that basket, `False` otherwise.

In [ ]:
from mlxtend.preprocessing import TransactionEncoder

# Initialize the TransactionEncoder
te = TransactionEncoder()

# Fit on transactions and transform into a boolean NumPy array
te_ary = te.fit(transactions).transform(transactions)

# Convert to a DataFrame with product names as column headers
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print("=== Encoded Transaction Matrix Summary ===")
print(f"Matrix Shape: {df_encoded.shape} (Rows: {df_encoded.shape[0]:,}, Columns: {df_encoded.shape[1]})")
print(f"Data Type:    {df_encoded.dtypes.iloc[0]}")
print(f"Memory Usage: {df_encoded.memory_usage().sum() / (1024 * 1024):.2f} MB")

# Display sample columns for the first 5 transactions
sample_cols = ["citrus fruit", "margarine", "whole milk", "yogurt", "coffee", "soda"]
print("\nSample Preview of Encoded Matrix:")
df_encoded[sample_cols].head()